# Secure and Reversible Data Embedding in Medical Images
## Using U-Net Segmentation and XOR Encryption

**Authors:** J. Dayanika, P. Jagatrayee, A. Mary Kavya  
**University:** Vignan's Foundation for Science, Technology and Research (VFSTR), Guntur, AP  
**Semester:** B.Tech 4-2 (Final Year Project)

---

### About This Project
A secure and reversible data embedding framework for brain tumor MRI images that integrates NLM denoising, U-Net-based tumor segmentation, LSB steganography, and XOR encryption — achieving lossless image restoration after data extraction.

### Dataset
**BraTS 2020** — [Download from Kaggle](https://www.kaggle.com/datasets/awsaf49/brats20-dataset-training-validation)

### Setup
1. Download and extract the BraTS dataset
2. Update the `DATA_DIR` variable in the first code cell
3. Run all cells sequentially

## 1. Setup & Data Loading

In [ ]:
# ============================================================
# Configure these paths based on your environment
# ============================================================

# Option 1: Google Colab (uncomment the two lines below)
# from google.colab import drive
# drive.mount('/content/drive')

# Option 2: Local environment
# Set DATA_DIR to the folder where you extracted the BraTS dataset
DATA_DIR = "/content/drive/MyDrive/brats_data"  # <-- Update this path

In [ ]:
import zipfile
import os

# If using a zip file, extract it first
# zip_path = "/path/to/your/archive.zip"  # Update this path
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(DATA_DIR)
# print("Extraction completed")

extract_path = DATA_DIR
print("Data directory:", extract_path)

## 2. Explore Dataset Folder Structure

In [ ]:
import os

# ================================
# STEP 3: Show folder structure
# ================================

for root, dirs, files in os.walk(extract_path):
    print("Folder:", root)
    print("Subfolders:", dirs[:3])
    print("Files:", files[:3])
    print("----------------------------")
    break

## 3. Visualize Sample MRI Slice

In [ ]:
# ================================
# STEP 4: Show Sample Image
# ================================

import cv2
import matplotlib.pyplot as plt
import nibabel as nib
import os

# Find first NIfTI image automatically
image_path = None

for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.endswith(".nii"): # Look for NIfTI files
            image_path = os.path.join(root, file)
            break
    if image_path:
        break

if image_path:
    try:
        # Load the NIfTI image
        img_nifti = nib.load(image_path).get_fdata()

        # Extract a central slice for visualization
        slice_img = img_nifti[:, :, img_nifti.shape[2] // 2]

        # Resize the slice
        slice_img_resized = cv2.resize(slice_img, (240, 240))

        # Display the grayscale image
        plt.imshow(slice_img_resized, cmap='gray')
        plt.title("Sample NIfTI Image Slice")
        plt.axis('off')
        plt.show()

        print("Image Path:", image_path)
        print("Output Size:", slice_img_resized.shape)

    except Exception as e:
        print(f"An error occurred while processing the NIfTI image: {e}")
else:
    print(f"Error: No NIfTI image found in the extracted path: {extract_path}")

## 4. Noise Estimation

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Normalize image to 0–255
norm_img = cv2.normalize(slice_img, None, 0, 255, cv2.NORM_MINMAX)
norm_img = norm_img.astype(np.uint8)

# ==========================
# METHOD-1: Standard Deviation Noise
# ==========================

noise_std = np.std(norm_img)

print("Noise Level (Standard Deviation):", noise_std)


# ==========================
# METHOD-2: Laplacian Noise (More Accurate)
# ==========================

laplacian = cv2.Laplacian(norm_img, cv2.CV_64F)

noise_laplacian = laplacian.var()

print("Noise Level (Laplacian Variance):", noise_laplacian)


# ==========================
# Noise Map Visualization
# ==========================

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(norm_img, cmap='gray')
plt.title("Original Image")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(laplacian, cmap='gray')
plt.title("Noise Map")
plt.axis("off")

plt.subplot(1,3,3)
plt.hist(norm_img.ravel(), bins=50)
plt.title("Pixel Intensity Distribution")

plt.show()

## 5. Noise Reduction & Filter Comparison

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


# Normalize
norm_img = cv2.normalize(slice_img, None, 0, 255, cv2.NORM_MINMAX)
norm_img = norm_img.astype(np.uint8)


# ===============================
# OPTIMIZED FILTERS
# ===============================

gaussian = cv2.GaussianBlur(norm_img, (7,7), 1.5)

median = cv2.medianBlur(norm_img, 7)

# bilateral = cv2.bilateralFilter(norm_img, 15, 150, 150)

# ⭐ STRONG Non Local Means
non_local = cv2.fastNlMeansDenoising(norm_img, None, 30, 7, 21)



# ===============================
# Noise Function
# ===============================

def noise(img):

    return np.std(img)


print("Noise BEFORE:", noise(norm_img))

print("Noise Gaussian:", noise(gaussian))

print("Noise Median:", noise(median))

# print("Noise Bilateral:", noise(bilateral))

print("Noise Non Local Means:", noise(non_local))



# ===============================
# Show images
# ===============================

plt.figure(figsize=(15,8))

plt.subplot(231)
plt.imshow(norm_img, cmap='gray')
plt.title("Original")

plt.subplot(232)
plt.imshow(gaussian, cmap='gray')
plt.title("Gaussian")

plt.subplot(233)
plt.imshow(median, cmap='gray')
plt.title("Median")

plt.subplot(235)
plt.imshow(non_local, cmap='gray')
plt.title("Non Local Means BEST")

plt.show()

## 6. Quantitative Noise Comparison Table

In [ ]:
import cv2
import numpy as np

print("----- Noise Comparison Table -----\n")

filters = {
    "Original": norm_img,
    "Gaussian": gaussian,
    "Median": median,
    "Non Local Means": non_local
}

for name, img in filters.items():

    std = np.std(img)

    lap = cv2.Laplacian(img, cv2.CV_64F).var()

    print(name)

    print("Standard Deviation:", std)

    print("Laplacian Variance:", lap)

    print("----------------------------")

## 7. Final Denoising with Non-Local Means

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


# Normalize image to float (IMPORTANT)
norm_img = cv2.normalize(slice_img, None, 0, 1, cv2.NORM_MINMAX)

norm_img = (norm_img * 255).astype(np.uint8)


# Apply Non Local Means (Optimized)
denoised_img = cv2.fastNlMeansDenoising(

    norm_img,

    None,

    h=25,              # strength (IMPORTANT)
    templateWindowSize=7,
    searchWindowSize=35

)


# Show comparison

plt.figure(figsize=(12,5))

plt.subplot(121)
plt.imshow(norm_img, cmap='gray')
plt.title("Original MRI")
plt.axis("off")

plt.subplot(122)
plt.imshow(denoised_img, cmap='gray')
plt.title("Denoised MRI (FINAL)")
plt.axis("off")

plt.show()


# Calculate PSNR (REAL QUALITY METRIC)

psnr = cv2.PSNR(norm_img, denoised_img)

print("PSNR Value:", psnr)

## 8. Grayscale Verification

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


# Use your best denoised image from previous step
# example:
gray_img = denoised_img.copy()


# Check if already grayscale

if len(gray_img.shape) == 3:
    gray_img = cv2.cvtColor(gray_img, cv2.COLOR_BGR2GRAY)


# Convert to uint8

gray_img = gray_img.astype(np.uint8)


# Show image

plt.imshow(gray_img, cmap='gray')
plt.title("Grayscale Image (Final for Embedding)")
plt.axis('off')
plt.show()


# Check details

print("Shape:", gray_img.shape)

print("Datatype:", gray_img.dtype)

print("Min pixel:", gray_img.min())

print("Max pixel:", gray_img.max())

## 9. LSB Data Embedding

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt


# Copy image

embed_img = gray_img.copy()


# -------------------------
# Patient Details
# -------------------------

patient_data = "Name:Jagatrayee;ID:12345;Diagnosis:Tumor"


# Convert text to binary

binary_data = ''.join(format(ord(i), '08b') for i in patient_data)

data_len = len(binary_data)

print("Total bits to embed:", data_len)


# -------------------------
# Embedding using LSB
# -------------------------

flat_img = embed_img.flatten()

for i in range(data_len):

    flat_img[i] = (flat_img[i] & 254) | int(binary_data[i])


embedded_img = flat_img.reshape(embed_img.shape)


# -------------------------
# Show output
# -------------------------

plt.figure(figsize=(10,5))

plt.subplot(121)
plt.imshow(embed_img, cmap='gray')
plt.title("Original Image")

plt.subplot(122)
plt.imshow(embedded_img, cmap='gray')
plt.title("Embedded Image")

plt.show()


# Save for next step

final_embedded_image = embedded_img.copy()

## 10. Tumor Region Segmentation (ROI Mask)

> **Note:** In production, this mask is generated by a trained U-Net model. For demonstration purposes, a circular placeholder mask is used.

In [ ]:
# Placeholder for final_tumor_mask
# In a real scenario, this would come from an image segmentation step.
final_tumor_mask = np.zeros(gray_img.shape, dtype=np.uint8)

# As a demonstration, let's create a small 'tumor' region in the center
# This is just for the code to run and demonstrate embedding.
center_x, center_y = gray_img.shape[1] // 2, gray_img.shape[0] // 2
radius = 20
cv2.circle(final_tumor_mask, (center_x, center_y), radius, 1, -1) # Draw a filled circle

## 11. ROI-Based Tumor Region Embedding

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2


# Use your grayscale denoised image
roi_image = gray_img.copy()

# Tumor mask from Step-1
mask = final_tumor_mask.copy()


# -------------------------
# Patient Data
# -------------------------

patient_data = "Name:Jagatrayee;ID:12345;Diagnosis:Tumor"

binary_data = ''.join(format(ord(i), '08b') for i in patient_data)

data_len = len(binary_data)

print("Bits to embed:", data_len)


# -------------------------
# Find tumor pixel locations
# -------------------------

tumor_positions = np.where(mask.flatten() == 1)[0]

print("Tumor pixel capacity:", len(tumor_positions))


# Check capacity

if data_len > len(tumor_positions):

    print("ERROR: Not enough tumor area")

else:

    print("Embedding possible")


# -------------------------
# Embed inside tumor region
# -------------------------

flat_img = roi_image.flatten()

for i in range(data_len):

    pos = tumor_positions[i]

    flat_img[pos] = (flat_img[pos] & 254) | int(binary_data[i])


embedded_img_roi = flat_img.reshape(roi_image.shape)


# -------------------------
# Show output
# -------------------------

plt.figure(figsize=(12,5))

plt.subplot(121)
plt.imshow(roi_image, cmap='gray')
plt.title("Original Image")

plt.subplot(122)
plt.imshow(embedded_img_roi, cmap='gray')
plt.title("Tumor-Region Embedded Image")

plt.show()


# Save

final_embedded_image = embedded_img_roi.copy()
embedding_positions = tumor_positions[:data_len]

## 12. Embedding Distortion Analysis

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt


original = embed_img.copy()
embedded = final_embedded_image.copy()


# Error map
error_map = cv2.absdiff(original, embedded)


# Binary error location
error_location = (error_map > 0).astype(np.uint8)


# Count errors
total_errors = np.sum(error_location)

print("Total Error Pixels:", total_errors)


# -------------------------
# ENHANCED VISUALIZATION
# -------------------------

plt.figure(figsize=(18,5))


plt.subplot(141)
plt.imshow(original, cmap='gray')
plt.title("Original")


plt.subplot(142)
plt.imshow(embedded, cmap='gray')
plt.title("Embedded")


plt.subplot(143)
plt.imshow(error_map, cmap='hot')   # heatmap
plt.title("Error Map (Heatmap)")


plt.subplot(144)
plt.imshow(error_location, cmap='gray', vmin=0, vmax=1)
plt.title("Error Location (Visible)")


plt.show()

## 13. Improved LSB Matching

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Copy original image
embedded_img = roi_image.copy()

# Flatten
flat_img = embedded_img.flatten()

# Store original bits
original_lsb_bits = []

# Improved embedding using LSB Matching
for i, pos in enumerate(embedding_positions):

    bit = int(binary_data[i])

    pixel = flat_img[pos]

    original_lsb_bits.append(pixel & 1)

    # Only modify if necessary
    if (pixel & 1) != bit:

        # LSB Matching (±1 instead of direct replace)
        if pixel == 0:
            pixel = pixel + 1

        elif pixel == 255:
            pixel = pixel - 1

        else:
            if np.random.rand() > 0.5:
                pixel = pixel + 1
            else:
                pixel = pixel - 1  # LSB matching: randomly ±1

    flat_img[pos] = pixel

# Reshape
embedded_img = flat_img.reshape(roi_image.shape)

# Show
plt.imshow(embedded_img, cmap='gray')
plt.title("Low Error Embedded Image")
plt.axis('off')
plt.show()

## 14. Quality Metrics (PSNR)

In [ ]:
import cv2

psnr = cv2.PSNR(roi_image, embedded_img)

print("PSNR:", psnr)

## 15. XOR Encryption

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Your embedded image
image_to_encrypt = final_embedded_image.copy()


# -----------------------------
# Encryption Key
# -----------------------------

key = 4236   # you can change (0–255)

np.random.seed(key)

# Generate key stream

key_stream = np.random.randint(0, 256, image_to_encrypt.shape, dtype=np.uint8)


# -----------------------------
# Encrypt Image
# -----------------------------

encrypted_image = np.bitwise_xor(image_to_encrypt, key_stream)


# -----------------------------
# Show Images
# -----------------------------

plt.figure(figsize=(12,5))

plt.subplot(121)
plt.imshow(image_to_encrypt, cmap='gray')
plt.title("Original Embedded Image")
plt.axis("off")


plt.subplot(122)
plt.imshow(encrypted_image, cmap='gray')
plt.title("Encrypted Image")
plt.axis("off")

plt.show()


# Save for next step

final_encrypted_image = encrypted_image.copy()
final_key_stream = key_stream.copy()

## 16. Base64 Encoding for Transmission

In [ ]:
import base64


# Convert encrypted image to bytes

encrypted_bytes = final_encrypted_image.tobytes()


# Convert bytes to Base64 text

encrypted_text = base64.b64encode(encrypted_bytes)


# Convert to string

encrypted_text = encrypted_text.decode('utf-8')


# Print first 500 characters

print(encrypted_text[:500])


# Save to file

with open("encrypted_image.txt", "w") as f:
    f.write(encrypted_text)


print("\nEncrypted image saved as encrypted_image.txt")

## 17. Decryption

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Decrypt using same key stream

decrypted_image = np.bitwise_xor(final_encrypted_image, final_key_stream)


# Show result

plt.figure(figsize=(12,5))

plt.subplot(121)
plt.imshow(final_encrypted_image, cmap='gray')
plt.title("Encrypted Image")
plt.axis("off")


plt.subplot(122)
plt.imshow(decrypted_image, cmap='gray')
plt.title("Decrypted Image")
plt.axis("off")

plt.show()


# Save

final_decrypted_image = decrypted_image.copy()

## 18. Decryption Verification

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


# Original embedded image (before encryption)
original = final_embedded_image.copy()


# Decrypted image
decrypted = final_decrypted_image.copy()


# --------------------------
# Noise Difference
# --------------------------

difference = cv2.absdiff(original, decrypted)


noise_value = np.std(difference)

print("Noise after Decryption:", noise_value)


# --------------------------
# PSNR
# --------------------------

psnr = cv2.PSNR(original, decrypted)

print("PSNR after Decryption:", psnr)


# --------------------------
# Show Difference Map
# --------------------------

plt.figure(figsize=(15,5))


plt.subplot(131)
plt.imshow(original, cmap='gray')
plt.title("Before Encryption")


plt.subplot(132)
plt.imshow(decrypted, cmap='gray')
plt.title("After Decryption")


plt.subplot(133)
plt.imshow(difference, cmap='hot')
plt.title("Noise Map")


plt.show()

## 19. Adaptive LSB Embedding with Noise Reduction

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt


# Use grayscale denoised image
original_img = gray_img.copy()

# Tumor mask from U-Net
mask = final_tumor_mask.copy()


# Patient data
patient_data = "Name:Jagatrayee;ID:12345;Diagnosis:Tumor"

binary_data = ''.join(format(ord(c), '08b') for c in patient_data)

data_len = len(binary_data)

print("Bits to embed:", data_len)


# Flatten
flat_img = original_img.flatten()
flat_mask = mask.flatten()


# Get tumor positions
tumor_positions = np.where(flat_mask == 1)[0]


# -----------------------------
# Adaptive LSB embedding
# Only modify if needed
# -----------------------------

modifications = 0

for i in range(data_len):

    pos = tumor_positions[i]

    bit = int(binary_data[i])

    current_lsb = flat_img[pos] & 1

    if current_lsb != bit:

        flat_img[pos] = (flat_img[pos] & 254) | bit

        modifications += 1


# Reshape
embedded_img = flat_img.reshape(original_img.shape)


print("Total modified pixels:", modifications)


# -----------------------------
# Show comparison
# -----------------------------

plt.figure(figsize=(12,5))

plt.subplot(121)
plt.imshow(original_img, cmap='gray')
plt.title("Original")

plt.subplot(122)
plt.imshow(embedded_img, cmap='gray')
plt.title("Noise Reduced Embedded")

plt.show()


# Save result
final_embedded_image = embedded_img.copy()


# -----------------------------
# Check Noise Map
# -----------------------------

difference = cv2.absdiff(original_img, embedded_img)

noise_value = np.std(difference)

psnr = cv2.PSNR(original_img, embedded_img)


print("Noise:", noise_value)

print("PSNR:", psnr)


plt.imshow(difference, cmap='hot')

plt.title("Reduced Noise Map")

plt.show()

## 20. Data Extraction

In [ ]:
# Decrypted image
extract_img = final_decrypted_image.copy()

# Flatten image
flat_img = extract_img.flatten()

# Extract using correct positions
extracted_bits = ""

for pos in embedding_positions:
    extracted_bits += str(flat_img[pos] & 1)


# Convert binary to text
extracted_text = ""

for i in range(0, len(extracted_bits), 8):

    byte = extracted_bits[i:i+8]

    extracted_text += chr(int(byte, 2))


print("Extracted Patient Data:")
print(extracted_text)

## 21. Lossless Image Restoration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Step 1: Use decrypted image
restored_img = final_decrypted_image.copy()

# Step 2: Flatten image
flat_img = restored_img.flatten()

# Step 3: Restore original LSB bits
for i, pos in enumerate(embedding_positions):

    # Clear LSB and replace with original bit
    flat_img[pos] = (flat_img[pos] & 254) | original_lsb_bits[i]


# Step 4: Reshape back to image
restored_img = flat_img.reshape(final_decrypted_image.shape)


# Step 5: Show restored image
plt.figure(figsize=(6,6))
plt.imshow(restored_img, cmap='gray')
plt.title("Perfectly Restored MRI Image")
plt.axis('off')
plt.show()